# Use Case — Tree-Planting Prioritization for Maximum Cooling Benefit

**Who this is for**  
Municipal foresters, parks-department planners, climate-adaptation teams, and NGOs allocating a limited tree-planting budget across many candidate sites.

**The scenario**  
You have two user datasets: an **existing tree inventory** and a **list of candidate planting locations**. The planting budget covers only a fraction of the candidates. You need to choose the candidates where each new tree delivers the **most cooling benefit per dollar**.

This notebook combines **both of your point layers** with **FortyGuard layers** to answer four questions:

1. **Where is it hottest?**  ← heatmap tells us which candidates sit in the worst heat
2. **Where are the canopy gaps?**  ← existing trees × candidates tells us where a new tree is not redundant
3. **Where is existing vegetation thinnest?**  ← satellite segmentation on top candidates
4. **Which candidates win the composite score?**  ← combining the three factors

Output: a ranked planting priority list CSV with an expected-benefit score per site.

> **Bring your own data.** Sample CSVs at `data/sample_existing_trees.csv` and `data/sample_candidate_planting_sites.csv`. Swap them — as long as each has `latitude`/`longitude` columns, everything downstream works.

**What makes this pattern different**: two user layers that interact. Neither of them alone is sufficient — planting a tree next to an existing tree is wasted; planting in a cool block is wasted; the highest-value plantings sit in the intersection of *hot* and *gappy*.

---

## Setup

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

import math
import pandas as pd
import folium
import matplotlib.pyplot as plt
from shapely.geometry import Point, shape

from fortyguard import FortyGuardClient
from fortyguard.samples import SAN_JOSE_POLYGON

client = FortyGuardClient()

AOI               = SAN_JOSE_POLYGON     # ~104 km² (~40 mi²) across central San Jose
STUDY_DATE        = '2024-07-15'
STUDY_HOUR        = '14:00'
GRANULARITY_M     = 100                  # 100 m keeps the heatmap tile count tractable over this AOI
TOP_N_TO_ANALYZE  = 5                    # satellite call budget — only runs on the top-N by initial rank

print(f'Authenticated to {client.base_url}')

---
## Step 1 — Load both user layers

### What you are doing
Reading the existing tree inventory and the candidate planting sites. Both are simple point layers.

### Why this matters
The pattern here is two user layers that interact. Neither one alone ranks candidates well — the interaction does.

In [ ]:
trees      = pd.read_csv(ROOT / 'data' / 'sample_existing_trees.csv')
candidates = pd.read_csv(ROOT / 'data' / 'sample_candidate_planting_sites.csv')
print(f'Existing trees : {len(trees)}')
print(f'Candidate sites: {len(candidates)}')
trees.head(), candidates.head()

---
## Step 2 — Visualize current canopy against candidates

### What you are doing
Rendering both layers on one map. Existing trees in green, sized by diameter. Candidates in yellow.

### Why this matters
Before any numbers are computed, this view alone already tells you which candidates sit in obvious canopy gaps. The quantitative analysis that follows formalizes what this map shows visually.

In [ ]:
center = [candidates['latitude'].mean(), candidates['longitude'].mean()]
fmap = folium.Map(location=center, zoom_start=15, tiles='cartodbpositron')
for _, t in trees.iterrows():
    folium.CircleMarker(
        location=[t.latitude, t.longitude],
        radius=max(4, t['diameter_cm'] / 15),
        color='#2d6b2d', fill=True, fill_color='#4caf50', fill_opacity=0.9, weight=1,
        popup=f"{t['tree_id']} — {t['species']} ({t['diameter_cm']} cm)",
    ).add_to(fmap)
for _, s in candidates.iterrows():
    folium.CircleMarker(
        location=[s.latitude, s.longitude],
        radius=6, color='#b08000', fill=True, fill_color='#f6c344', fill_opacity=0.9, weight=1,
        popup=f"{s['site_id']} — {s['block']} (pit: {s['available_pit_size']})",
    ).add_to(fmap)
fmap

---
## Step 3 — Generate the heat layer and join temperature to candidates

### What you are doing
One heatmap, then spatial-join the tile temperature onto every candidate site.

### Why this matters
Temperature is the first of three signals in the cooling-benefit composite. Hot candidates earn more of the budget — but only if the other two signals (gap and vegetation) also align.

In [ ]:
heatmap = client.create_heatmap(
    polygon_aoi=AOI, start_date=STUDY_DATE, start_time=STUDY_HOUR,
    filter_type=1, granularity=GRANULARITY_M,
)
map_data = heatmap['result'].get('map_data') or {}
features = map_data.get('features', []) if isinstance(map_data, dict) else []
tile_polys = [(shape(f['geometry']), f['properties'].get('temperature')) for f in features]

def _temp_at(lat, lon):
    p = Point(lon, lat)
    for poly, t in tile_polys:
        if poly.contains(p): return t
    return min(tile_polys, key=lambda pt: pt[0].centroid.distance(p))[1] if tile_polys else None

candidates['temperature_c'] = candidates.apply(
    lambda r: _temp_at(r.latitude, r.longitude), axis=1)
candidates[['site_id', 'block', 'temperature_c']].head()

---
## Step 4 — Measure the canopy gap at each candidate

### What you are doing
For each candidate, computing the distance to the nearest existing tree. The farther away, the larger the canopy gap that a new tree fills.

### Why this matters
This is the interaction between your two user layers. A candidate 4 m from a large existing tree delivers almost no additional shade. A candidate 60 m from the nearest tree fills a real gap in the network. A purely thermal ranking cannot see this; yours will.

In [ ]:
# Approximate distance in meters using an equirectangular projection — fine for small areas.
def _meters(lat1, lon1, lat2, lon2):
    mean_lat_rad = math.radians((lat1 + lat2) / 2)
    dy = (lat2 - lat1) * 111_000
    dx = (lon2 - lon1) * 111_000 * math.cos(mean_lat_rad)
    return math.hypot(dx, dy)

def _nearest_tree(row):
    distances = trees.apply(
        lambda t: _meters(row.latitude, row.longitude, t.latitude, t.longitude), axis=1)
    idx = distances.idxmin()
    return pd.Series({
        'nearest_tree_id': trees.loc[idx, 'tree_id'],
        'nearest_tree_m': round(distances.min(), 1),
    })

candidates = candidates.join(candidates.apply(_nearest_tree, axis=1))
candidates[['site_id', 'block', 'temperature_c', 'nearest_tree_id', 'nearest_tree_m']].head()

---
## Step 5 — Characterize surrounding vegetation on the top-N candidates

### What you are doing
Running satellite segmentation on the top-N candidates (ranked so far by temperature × gap). We record the vegetation fraction in their immediate surroundings.

### Why this matters
A candidate sitting in a block that is already 30 % vegetation benefits less from one more tree than a block at 3 %. This is the third signal in the composite — and it also keeps the satellite call budget small (we only pay for the top candidates, not all of them).

In [ ]:
def _mm(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else s * 0

candidates['temp_norm']   = _mm(candidates['temperature_c'])
candidates['gap_norm']    = _mm(candidates['nearest_tree_m'])
candidates['preliminary_score'] = (0.5 * candidates['temp_norm']
                                    + 0.5 * candidates['gap_norm'])

shortlist = candidates.sort_values('preliminary_score', ascending=False).head(TOP_N_TO_ANALYZE).copy()

VEGGIE = {'vegetation', 'tree', 'trees', 'grass', 'greenery', 'park'}
def _veg_pct(segments):
    total = 0.0
    for cls, pct in (segments or {}).items():
        if any(k in cls.lower() for k in VEGGIE):
            try: total += float(pct)
            except (TypeError, ValueError): pass
    return round(total, 1)

vegs = []
for _, r in shortlist.iterrows():
    print(f"  satellite: {r.site_id} — {r.block}")
    sat = client.satellite_segmentation(
        latitude=r.latitude, longitude=r.longitude,
        start_date=STUDY_DATE, start_time=STUDY_HOUR,
        filter_type=1, granularity=GRANULARITY_M, verbose=False)
    segs = sat['result'].get('segmentation', {}).get('segments', {}) or {}
    vegs.append(_veg_pct(segs))
shortlist['vegetation_pct'] = vegs
shortlist[['site_id', 'block', 'temperature_c', 'nearest_tree_m', 'vegetation_pct']]

---
## Step 6 — Composite cooling-benefit score

### What you are doing
Combining the three signals on the shortlist into one priority score:

- **40 %** thermal severity (tile temperature)
- **35 %** canopy gap (distance to nearest existing tree)
- **25 %** vegetation deficit (`1 - vegetation_pct/100`)

### Why this matters
These weights encode a defensible philosophy: hot spots are the primary target, but we do not spend on redundant plantings, and we prefer sites where existing green-infrastructure is scarce. The weights are tunable — change them to reflect local policy (e.g., environmental-justice zones get a population-overlay weight).

In [ ]:
shortlist['temp_norm'] = _mm(shortlist['temperature_c'])
shortlist['gap_norm']  = _mm(shortlist['nearest_tree_m'])
shortlist['veg_deficit_norm'] = _mm(100 - shortlist['vegetation_pct'])

shortlist['priority_score'] = (0.40 * shortlist['temp_norm']
                                + 0.35 * shortlist['gap_norm']
                                + 0.25 * shortlist['veg_deficit_norm']).round(3)
shortlist = shortlist.sort_values('priority_score', ascending=False).reset_index(drop=True)
shortlist.insert(0, 'rank', shortlist.index + 1)
shortlist[['rank', 'site_id', 'block', 'available_pit_size', 'temperature_c',
           'nearest_tree_m', 'vegetation_pct', 'priority_score']]

---
## Step 7 — Planting priority list

### What you are doing
Producing the final action list. Each row is a candidate, each column is evidence. The `priority_score` column is defensible at budget review because every input is traceable.

### Why this matters
Foresters lose funding when their prioritization looks arbitrary. This output is *not* arbitrary — temperature comes from the API, gap comes from the user inventory, vegetation comes from the API, weights are explicit. The council can argue the weights and see the result move.

In [ ]:
def _confidence(r):
    if r['priority_score'] >= 0.67: return 'plant (high confidence)'
    if r['priority_score'] >= 0.40: return 'plant (secondary)'
    return 'defer'

shortlist['recommendation'] = shortlist.apply(_confidence, axis=1)

out_cols = ['rank', 'site_id', 'block', 'available_pit_size',
            'temperature_c', 'nearest_tree_m', 'vegetation_pct',
            'priority_score', 'recommendation']
priority_list = shortlist[out_cols]
priority_list

In [ ]:
out = ROOT / 'outputs' / 'tree_planting_priority_list.csv'
out.parent.mkdir(parents=True, exist_ok=True)
priority_list.to_csv(out, index=False)
print(f'Saved priority list to {out}')

# Final map: existing canopy in green, ranked candidates sized by score.
fmap = folium.Map(location=center, zoom_start=15, tiles='cartodbpositron')
for _, t in trees.iterrows():
    folium.CircleMarker(
        location=[t.latitude, t.longitude],
        radius=max(4, t['diameter_cm'] / 15),
        color='#2d6b2d', fill=True, fill_color='#4caf50', fill_opacity=0.7, weight=1,
    ).add_to(fmap)
for _, s in shortlist.iterrows():
    folium.CircleMarker(
        location=[s.latitude, s.longitude],
        radius=6 + s['priority_score'] * 10,
        color='black', weight=1,
        fill=True, fill_color='#d73027', fill_opacity=0.85,
        popup=(f"#{s['rank']} {s['site_id']} — {s['block']}<br/>"
               f"score: {s['priority_score']:.2f} ({s['recommendation']})<br/>"
               f"temp: {s.temperature_c:.1f} °C, gap: {s['nearest_tree_m']:.0f} m, "
               f"veg: {s['vegetation_pct']}%"),
    ).add_to(fmap)
fmap

---
## Wrap-up

Starting from two point layers (existing trees, candidate sites) you now have:

| Artifact | Audience |
|----------|----------|
| Candidate table with tile temperature | GIS / arboriculture team |
| Nearest-tree distance per candidate (canopy gap) | Spatial analysis |
| Vegetation fraction in surroundings (top-N) | Ecological assessment |
| Composite priority score with recommendation | Budget committee |
| Final priority-list CSV + map | Operations / contractors |

Every input is explicit and every weight is tunable. That is what makes this defensible in a council hearing — and what lets you rerun the scoring with updated priorities (environmental-justice overlays, particular species suitability) without rebuilding the pipeline.

**Apply this pattern to adjacent use cases**: any two-layer user input with interaction works here — existing bus stops × candidate new stops, existing cooling centers × candidate sites, existing EV chargers × candidate locations. The workflow — *current network × candidates × our thermal layer → cooling-benefit ranking* — transfers directly.